# RPG Recommender — Interactive Demo

This notebook demonstrates the trained RPG model.

**Workflow:**
1. Load the model from a checkpoint
2. Search the item catalog by keyword to find valid ASINs
3. Build a history by picking items from the search results
4. Call `recommend()` to get the top-K predicted next items

## Step 1 — Load the model

Set the checkpoint path and toggle `subcategory_injection` to match the model you trained.

In [ ]:
import sys
sys.path.insert(0, '..')   # so Python can find the genrec package

from inference import RecommenderInference

# ── CONFIG — change these to match your run ──────────────────────────────────
CHECKPOINT_PATH = 'checkpoints/RPG_Beauty.pth'   # path to your .pth file
SUBCATEGORY_INJECTION = False                      # True = injected model, False = baseline

CONFIG = {
    'category': 'Beauty',
    'metadata': 'sentence',
    'prune_items': None,           # set to 1000 if you pruned, else None
    'subcategory_injection': SUBCATEGORY_INJECTION,
}
# ─────────────────────────────────────────────────────────────────────────────

rec = RecommenderInference(
    model_name='RPG',
    dataset_name='AmazonReviews2014',
    checkpoint_path=CHECKPOINT_PATH,
    config_dict=CONFIG,
)
print('Model loaded successfully!')

## Step 2 — Search the catalog

Since the model only knows items it was trained on, you **must use ASINs from the catalog**.  
Search by keyword below to find valid items and their ASINs.

In [ ]:
# Change this keyword to browse different product types
# e.g. 'shampoo', 'lipstick', 'moisturizer', 'nail polish', 'perfume'
SEARCH_KEYWORD = 'shampoo'

results = rec.search_items_by_keyword(SEARCH_KEYWORD, max_results=10)

print(f'Items matching "{SEARCH_KEYWORD}":\n')
print(f'{"#":<4} {"ASIN":<14} {"Description (first 120 chars)"}')
print('-' * 80)
for i, r in enumerate(results, 1):
    print(f'{i:<4} {r["asin"]:<14} {r["meta"][:120]}')

## Step 3 — Build a user history

Copy ASINs from the search results above and paste them into `USER_HISTORY` below.  
Order matters — put them **chronological order, most recent last**.

In [ ]:
# Paste ASINs from the search results above
# You need at least 2 items for the model to have meaningful context
USER_HISTORY = [
    'B001234567',   # ← replace with real ASINs from search results
    'B007654321',
    'B009876543',
]

# Verify they are in the catalog
print('Verifying history items:')
for asin in USER_HISTORY:
    info = rec.get_item_info(asin)
    status = '✅' if info['item_id'] is not None else '❌ NOT IN CATALOG'
    print(f'  {status}  {asin}  —  {info["meta"][:100]}')

## Step 4 — Get recommendations

In [ ]:
TOPK = 10

recs = rec.recommend(USER_HISTORY, topk=TOPK)

print(f'Top-{TOPK} recommendations for the given history:\n')
print(f'{"Rank":<6} {"ASIN":<14} {"Description (first 150 chars)"}')
print('-' * 90)
for r in recs:
    print(f'#{r["score_rank"]:<5} {r["asin"]:<14} {r["meta"][:150]}')

## Step 5 — Compare baseline vs injected model (optional)

Load both checkpoints and run the same history through both to show the difference.

In [ ]:
BASELINE_CKPT = 'checkpoints/RPG_Beauty_baseline.pth'
INJECTED_CKPT = 'checkpoints/RPG_Beauty_injected.pth'

rec_baseline = RecommenderInference(
    model_name='RPG', dataset_name='AmazonReviews2014',
    checkpoint_path=BASELINE_CKPT,
    config_dict={**CONFIG, 'subcategory_injection': False}
)
rec_injected = RecommenderInference(
    model_name='RPG', dataset_name='AmazonReviews2014',
    checkpoint_path=INJECTED_CKPT,
    config_dict={**CONFIG, 'subcategory_injection': True}
)

recs_base = rec_baseline.recommend(USER_HISTORY, topk=TOPK)
recs_inj  = rec_injected.recommend(USER_HISTORY, topk=TOPK)

# Side-by-side comparison
print(f'{"Rank":<6} {"BASELINE ASIN":<16} {"INJECTED ASIN":<16}')
print('-' * 60)
for b, i in zip(recs_base, recs_inj):
    marker = '  ←same' if b['asin'] == i['asin'] else ''
    print(f'#{b["score_rank"]:<5} {b["asin"]:<16} {i["asin"]:<16}{marker}')

print('\n--- Baseline descriptions ---')
for r in recs_base[:3]:
    print(f'  [{r["asin"]}] {r["meta"][:120]}')

print('\n--- Injected descriptions ---')
for r in recs_inj[:3]:
    print(f'  [{r["asin"]}] {r["meta"][:120]}')